# Homework 3: Recurrent Neural Networks (100 points)

### Overview

We now move from image recognition to natural language processing. For this assignment, we will work with a common sentiment analysis task using the IMDB dataset. This set consists of review-label pairs, where the task is to predict whether the text of the given movie review is positive or negative, a binary classification.

### RNN Architecture

You will be comparing four different recurrent neural network architectures: a standard RNN, a Gated Recurrent Unit (GRU), a standard Long Short-Term Memory (LSTM), and a bidirectional LSTM. 

Note that a GRU/LSTM cell _is_ an RNN cell, but we will refer to an RNN in the code and questions below as the most basic RNN.

### Your Task

At the bottom of this notebook file, there are three short answer questions testing your understanding of this neural network architecture. 

Below each question is a cell with the text “Type Markdown and LaTex.” Double-click the cell and type your response to the question. Save your responses by clicking on the floppy disk icon or choosing File - Save and Checkpoint.

After responding to the questions, click the submit button at the top of the notebook to submit your assignment for grading.

In [1]:
import torch
import torch.nn as nn
import pickle

In [2]:
torch.manual_seed(0)
torch.set_num_threads(4)
torch.set_num_interop_threads(4)

In [3]:
from mads.lib.path import assets
root_dir = str(assets.find("week3"))

reviewVocabVectors = pickle.load(open(root_dir + '/reviewVocabVectors', 'rb'))
trainIterator = pickle.load(open(root_dir + '/trainIterator', 'rb'))
testIterator = pickle.load(open(root_dir + '/testIterator', 'rb'))

In [4]:
# Model/training hyperparameters
embeddingSize = 100
hiddenSize = 10
dropoutRate = 0.5
numEpochs = 5
vocabSize = 20002
pad = 1
unk = 0

class MyRNN(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.name = model

        # Flag whether the selected architecture is an LSTM variant.
        self.LSTM = (model == 'LSTM' or model == 'BiLSTM')
        # BiLSTM uses two directions; the others use a single direction.
        self.bidir = (model == 'BiLSTM')

        # Convert token ids into dense word vectors.
        self.embed = nn.Embedding(vocabSize, embeddingSize, padding_idx=pad)

        # Select the recurrent layer based on the requested architecture.
        if model == 'RNN': 
            self.rnn = nn.RNN(embeddingSize, hiddenSize)
        elif model == 'GRU': 
            self.rnn = nn.GRU(embeddingSize, hiddenSize)
        else: 
            self.rnn = nn.LSTM(embeddingSize, hiddenSize, bidirectional=self.bidir)

        # The BiLSTM outputs hidden states from both directions, so its
        # final feature size is doubled before the classification layer.
        self.dense = nn.Linear(hiddenSize * (2 if self.bidir else 1), 1)
        self.dropout = nn.Dropout(dropoutRate)

    def forward(self, text, textLengths):
        # Apply embeddings and dropout regularization.
        embedded = self.dropout(self.embed(text))

        # Pack padded sequences so the recurrent model ignores pad tokens.
        packedEmbedded = nn.utils.rnn.pack_padded_sequence(embedded, textLengths)

        # Run the packed sequence through the selected recurrent layer.
        if self.LSTM:
            packedOutput, (hidden, cell) = self.rnn(packedEmbedded)
        else:
            packedOutput, hidden = self.rnn(packedEmbedded)

        # For a BiLSTM, concatenate the last forward and backward hidden states.
        if self.bidir:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1))
        else:
            hidden = self.dropout(hidden[-1,:,:])

        # Produce a single logit for binary sentiment classification.
        return self.dense(hidden)

In [ ]:
# Instantiate one model of each architecture requested in the assignment.
basicRNN = MyRNN(model='RNN')   # <--1)
GRU = MyRNN(model='GRU')        # <--2)
LSTM = MyRNN(model='LSTM')      # <--3)
BiLSTM = MyRNN(model='BiLSTM')  # <--4)

# Keep the models together so the same training/evaluation code can loop over them.
models = [basicRNN, GRU, LSTM, BiLSTM]

In [6]:
# Initialize all embedding layers with the provided pretrained word vectors.
for model in models:
    if model is None:
        continue
    model.embed.weight.data.copy_(reviewVocabVectors)

    # Force the unknown-token and padding-token embeddings to zero vectors
    # so they do not inject arbitrary information into the sequence model.
    model.embed.weight.data[unk] = torch.zeros(embeddingSize)
    model.embed.weight.data[pad] = torch.zeros(embeddingSize)

In [7]:
# BCEWithLogitsLoss combines a sigmoid layer with binary cross-entropy,
# which is appropriate because the task is binary sentiment classification.
criterion = nn.BCEWithLogitsLoss()

def batchAccuracy(preds, targets):
    # Convert logits to class predictions using 0 as the decision boundary.
    roundedPreds = (preds >= 0)
    return (roundedPreds == targets).sum().item() / len(preds)

In [ ]:
# Training

# Put each model in training mode so dropout is active during learning.
for model in models: 
    if model is not None:
        model.train()

# Train each architecture separately using the same optimizer/settings
# so their validation accuracies can be compared fairly.
for model in models:
    if model is None:
        continue

    torch.manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters())
    for epoch in range(numEpochs):
        epochLoss = 0
        for batch in trainIterator:
            optimizer.zero_grad()
            text, textLen = batch[0]
            predictions = model(text, textLen).squeeze(1)
            loss = criterion(predictions, batch[1])
            loss.backward()
            optimizer.step()
            epochLoss += loss.item()
        print(f'Model: {model.name}, Epoch: {epoch + 1}, Train Loss: {epochLoss / len(trainIterator)}')
    print()

Model: RNN, Epoch: 1, Train Loss: 0.7023946513300356


In [ ]:
# Evaluation

# Switch to evaluation mode so dropout is disabled at validation time.
for model in models: 
    if model is not None:
        model.eval()

with torch.no_grad():

    for model in models:

        if model is None:
            continue

        accuracy = 0.0
        for batch in testIterator:
            text, textLen = batch[0]
            predictions = model(text, textLen).squeeze(1)
            loss = criterion(predictions, batch[1])
            acc = batchAccuracy(predictions, batch[1])
            accuracy += acc

        # Report the mean validation accuracy for this architecture.
        print('Model: {}, Validation Accuracy: {}%'.format(model.name, accuracy / len(testIterator) * 100))

## Homework Questions

**To make sure your code produces consistent results, it is advisable to click "Kernel -> Restart & Run All" every time you want to run your code.**

```
Model: RNN, Validation Accuracy: 69.40776854219948%
Model: GRU, Validation Accuracy: 82.62707800511508%
Model: LSTM, Validation Accuracy: 85.89114450127879%
Model: BiLSTM, Validation Accuracy: 83.81873401534527%
```

### Question 1: Coding (50 points)

First, run the code given above to assess accuracy of the default RNN model. 

Next, you will need to construct three other model types (GRU, LSTM, BiLSTM) for comparison purposes. Follow the comments in box 5 to initialize the three other model types then rerun the code with all models enabled.

Finally, compare the accuracies of all four models (the accuracy of the default RNN should not change from the initial run). Explain your results. In doing so, overview the advantages of the best performing architecture.

Among the four models, the **LSTM** performed best on the validation set with an accuracy of **85.89%**, followed by **BiLSTM** at **83.82%**, **GRU** at **82.63%**, and the basic **RNN** at **69.41%**.

These results are consistent with what we expect from sequence models. A standard **RNN** often struggles with long-range dependencies because repeated updates over many time steps can lead to the **vanishing gradient problem**. That makes it harder for the model to retain important information from earlier parts of a long movie review, which likely explains its much lower validation accuracy.

Both the **GRU** and **LSTM** improve on the vanilla RNN by using gating mechanisms that help the network decide what information to keep, update, or forget across time. This gives them a stronger ability to preserve useful context throughout a sequence. In this notebook, both gated models clearly outperform the basic RNN.

The **LSTM** achieved the highest validation accuracy, suggesting that its memory cell and gating structure were especially effective for this sentiment classification task. The **BiLSTM** also performed well because it can incorporate information from both directions of the sequence, but here it still did not surpass the one-directional LSTM.

Overall, the results suggest that architectures designed to better manage sequential memory substantially outperform a vanilla RNN on this problem, and in this case the **LSTM** was the strongest model of the four.

### Question 2: LSTM Gates (30 points)

LSTMs improve upon the naive RNN architecture by adding a series of gates instead of a simple matrix-vector computation. Name the gates and explain each of their functions.

An LSTM uses three main gates plus the cell-state update:

1. **Forget gate**  
   The forget gate decides what information from the previous cell state should be kept or discarded. It takes the current input and previous hidden state, then outputs values between 0 and 1. Values near 0 mean the information should mostly be forgotten, while values near 1 mean it should mostly be retained.

2. **Input gate**  
   The input gate controls what new information should be written into the cell state. It determines how much of the candidate information computed at the current time step gets added to memory.

3. **Output gate**  
   The output gate decides what part of the updated cell state should be exposed as the hidden state for the current step. That hidden state is then passed forward to the next time step and can also be used by later layers.

4. **Candidate cell update**  
   Alongside the gates, the LSTM computes a candidate update for the cell state. The forget and input gates work together to combine the old memory with this new candidate information.

Together, these components let the LSTM selectively remember important long-term information, ignore irrelevant content, and expose only the most useful information at each step. This is why LSTMs are much better than vanilla RNNs at modeling long sequences.

### Question 3: Applications (20 points)

LSTMs are used across many different fields, from music generation to sentiment classification to text generation. Where could they be useful in your life, whether at home, for your family, or in the workplace? Give a specific problem or application for an LSTM model that was not covered in the course slides (**though it can be related to the applications covered in the slides**). Then, with your application in mind, specifically identify the input to your model, the output from your model, and an applicable loss function. 

(As an optional extension, try implementing your LSTM on your own using the code framework given in this homework!)

One useful LSTM application in my workplace would be **predicting whether a patient is at risk of missing an upcoming appointment** based on their recent interaction history. In a healthcare setting, missed appointments can delay care, reduce clinic efficiency, and make it harder for patients to receive timely services. Because patient behavior often depends on patterns over time, an LSTM would be a good fit for this problem.

**Specific problem/application:**  
Use an LSTM to predict whether a patient will miss an upcoming appointment so staff can intervene early with reminder calls, transportation support, or follow-up outreach.

**Input to the model:**  
The input would be a **time-ordered sequence of patient events** leading up to the appointment. Each time step could include features such as:
- number of days until the appointment
- whether the patient confirmed the visit
- prior attendance or no-show history
- reminder call or text outcomes
- transportation barriers
- referral or authorization status
- recent communication activity with staff

Because these events occur over time, an LSTM could learn patterns in the sequence that may signal rising no-show risk.

**Output from the model:**  
The output would be a **single probability** that the patient will miss the appointment. This can be interpreted as a binary classification output:
- 1 = likely to miss the appointment
- 0 = likely to attend the appointment

**Applicable loss function:**  
An appropriate loss function would be **binary cross-entropy loss** (for example, `BCEWithLogitsLoss` in PyTorch), because the model is predicting the probability of one of two classes.

This is a good use case for an LSTM because the order and timing of patient interactions matter, and those sequential patterns may be important for identifying which patients need additional support before care is missed.